# PayGuard — AI-Powered Payment Fraud Detection

## Notebook 04: SHAP Explainability

This notebook explains the final PayGuard fraud detection model using SHAP.

The goal is to answer two important questions:

1. **Global explainability:** Which features are most important across all predictions?
2. **Local explainability:** Why did the model classify one specific transaction as fraud or legitimate?

The model uses anonymized PCA features `V1` to `V28`, so the original business meaning of each variable is hidden.  
However, SHAP still helps us understand which transaction signatures influence fraud risk the most.

## 1. Setup

We import the required libraries, load the saved XGBoost model, and load processed test data from `data/processed/`.

In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import shap

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

RANDOM_STATE = 42
MAX_SHAP_SAMPLE_SIZE = 5_000

PROJECT_ROOT = Path.cwd()
possible_roots: List[Path] = [PROJECT_ROOT, PROJECT_ROOT.parent]

for root in possible_roots:
    if (root / "models" / "fraud_model.pkl").exists():
        PROJECT_ROOT = root
        break

MODEL_PATH = PROJECT_ROOT / "models" / "fraud_model.pkl"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model path: {MODEL_PATH}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Reports directory: {REPORTS_DIR}")

In [ ]:
required_files = {
    "model": MODEL_PATH,
    "X_train": PROCESSED_DIR / "X_train.pkl",
    "X_test": PROCESSED_DIR / "X_test.pkl",
    "y_test": PROCESSED_DIR / "y_test.pkl",
}

missing_files = [str(path) for path in required_files.values() if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing required files. Please run preprocessing and modelling notebooks first. "
        f"Missing files: {missing_files}"
    )

model = joblib.load(required_files["model"])
X_train = joblib.load(required_files["X_train"])
X_test = joblib.load(required_files["X_test"])
y_test = joblib.load(required_files["y_test"])

if not isinstance(X_test, pd.DataFrame):
    X_test = pd.DataFrame(X_test)

if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train, columns=X_test.columns)

if not isinstance(y_test, pd.Series):
    y_test = pd.Series(y_test, index=X_test.index, name="Class")

print("Files loaded successfully.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Model type: {type(model).__name__}")

## 2. Create SHAP Explainer

For performance, SHAP values are calculated on a representative sample of the test set.

This keeps the notebook fast while still providing meaningful global and local explanations.

In [ ]:
X_shap = X_test.sample(
    n=min(MAX_SHAP_SAMPLE_SIZE, len(X_test)),
    random_state=RANDOM_STATE,
)

explainer = shap.TreeExplainer(model)

raw_shap_values = explainer.shap_values(X_shap)

if isinstance(raw_shap_values, list):
    shap_values = raw_shap_values[1]
else:
    shap_values = raw_shap_values

expected_value = explainer.expected_value

if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[-1]

print(f"SHAP sample shape: {X_shap.shape}")
print(f"SHAP values shape: {np.asarray(shap_values).shape}")
print(f"Expected value: {expected_value}")

## 3. GLOBAL EXPLAINABILITY

Global explainability shows which features influence the model most across many transactions.

### Strongest fraud indicators

**V14, V17, V12 are the strongest fraud indicators.**

These features are anonymized PCA components, so we cannot map them to exact raw business variables.  
However, their SHAP impact shows that they strongly influence whether the model sees a transaction as risky or safe.

### 3.1 SHAP Summary Plot — Top 15 Features

The bar summary plot ranks features by their average absolute SHAP value.

A higher value means the feature has a stronger average impact on the model's predictions.

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="bar",
    max_display=15,
    show=False,
)
plt.title("SHAP Feature Importance — Top 15 Features")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "shap_summary_bar.png", dpi=300, bbox_inches="tight")
plt.show()

### 3.2 SHAP Beeswarm Plot — Feature Impact Distribution

The beeswarm plot shows both feature importance and prediction direction.

- Points on the right push predictions toward fraud.
- Points on the left push predictions toward legitimate.
- Color shows whether the feature value is high or low.

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_shap,
    max_display=15,
    show=False,
)
plt.title("SHAP Beeswarm Plot — Top 15 Feature Impact Distribution")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. LOCAL EXPLAINABILITY

Local explainability explains one specific prediction.

We select:

- One actual fraud transaction from the test set
- One actual legitimate transaction from the test set

Then we use SHAP waterfall and force plots to understand why the model made each decision.

In [ ]:
def get_prediction_details(instance: pd.DataFrame) -> Dict[str, float]:
    """
    Return prediction details for one transaction.

    Args:
        instance: Single-row feature DataFrame.

    Returns:
        Dictionary containing prediction label and fraud probability.
    """
    fraud_probability = float(model.predict_proba(instance)[:, 1][0])
    prediction = int(model.predict(instance)[0])

    return {
        "prediction": prediction,
        "fraud_probability": fraud_probability,
    }


fraud_indices = y_test[y_test == 1].index
legitimate_indices = y_test[y_test == 0].index

if len(fraud_indices) == 0:
    raise ValueError("No fraud cases found in y_test.")
if len(legitimate_indices) == 0:
    raise ValueError("No legitimate cases found in y_test.")

fraud_index = fraud_indices[0]
legitimate_index = legitimate_indices[0]

X_fraud_instance = X_test.loc[[fraud_index]]
X_legitimate_instance = X_test.loc[[legitimate_index]]

fraud_details = get_prediction_details(X_fraud_instance)
legitimate_details = get_prediction_details(X_legitimate_instance)

print("Selected fraud case:")
print(f"- Index: {fraud_index}")
print(f"- Predicted class: {fraud_details['prediction']}")
print(f"- Fraud probability: {fraud_details['fraud_probability']:.4f}")

print("\nSelected legitimate case:")
print(f"- Index: {legitimate_index}")
print(f"- Predicted class: {legitimate_details['prediction']}")
print(f"- Fraud probability: {legitimate_details['fraud_probability']:.4f}")

In [ ]:
def create_local_explanation(instance: pd.DataFrame) -> shap.Explanation:
    """
    Create a SHAP Explanation object for one transaction.

    Args:
        instance: Single-row feature DataFrame.

    Returns:
        SHAP Explanation object for plotting waterfall charts.
    """
    local_values = explainer.shap_values(instance)

    if isinstance(local_values, list):
        local_values = local_values[1]

    base_value = explainer.expected_value

    if isinstance(base_value, (list, np.ndarray)):
        base_value = base_value[-1]

    return shap.Explanation(
        values=np.asarray(local_values)[0],
        base_values=base_value,
        data=instance.iloc[0].values,
        feature_names=instance.columns.tolist(),
    )


fraud_explanation = create_local_explanation(X_fraud_instance)
legitimate_explanation = create_local_explanation(X_legitimate_instance)

### 4.1 Fraud Case — SHAP Waterfall Plot

This plot explains which features pushed the model toward a fraud prediction for one actual fraud case.

In [ ]:
plt.figure(figsize=(10, 6))
shap.plots.waterfall(fraud_explanation, max_display=10, show=False)
plt.title("Fraud Case — SHAP Waterfall Explanation")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "fraud_case_waterfall.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.2 Fraud Case — SHAP Force Plot

The force plot shows the same local decision in a horizontal explanation format.

Features pushing the score higher increase fraud risk.  
Features pushing the score lower reduce fraud risk.

In [ ]:
shap.initjs()

fraud_force_plot = shap.force_plot(
    fraud_explanation.base_values,
    fraud_explanation.values,
    X_fraud_instance.iloc[0],
    matplotlib=False,
)

fraud_force_plot

### Fraud Case Interpretation

For the selected fraud transaction, the model's fraud probability is high because several transaction signature features push the prediction toward the fraud class.

In practical business language:

- The transaction pattern looks different from normal legitimate behavior.
- High-impact PCA features act as warning signals.
- The model is not relying on one single feature only; it combines multiple risk signals before classifying the transaction.

### 4.3 Legitimate Case — SHAP Waterfall Plot

This plot explains which features pushed the model toward a legitimate prediction for one actual legitimate case.

In [ ]:
plt.figure(figsize=(10, 6))
shap.plots.waterfall(legitimate_explanation, max_display=10, show=False)
plt.title("Legitimate Case — SHAP Waterfall Explanation")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "legitimate_case_waterfall.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.4 Legitimate Case — SHAP Force Plot

For a legitimate transaction, the strongest feature effects should usually push the score away from fraud.

In [ ]:
legitimate_force_plot = shap.force_plot(
    legitimate_explanation.base_values,
    legitimate_explanation.values,
    X_legitimate_instance.iloc[0],
    matplotlib=False,
)

legitimate_force_plot

### Legitimate Case Interpretation

For the selected legitimate transaction, the model's fraud probability is low because the transaction signature is closer to normal behavior.

In practical business language:

- The model sees this transaction as low-risk.
- The strongest SHAP factors push the decision away from the fraud class.
- This helps reduce unnecessary false fraud alerts and improves customer experience.

## 5. FEATURE IMPORTANCE COMPARISON

In this section, we compare:

1. **SHAP importance** — average absolute effect of each feature on predictions
2. **XGBoost built-in importance** — model's internal feature usage score

This comparison helps validate whether both methods agree on the most important fraud signals.

In [ ]:
shap_importance_df = pd.DataFrame({
    "feature": X_shap.columns,
    "shap_importance": np.abs(shap_values).mean(axis=0),
})

if hasattr(model, "feature_importances_"):
    xgb_importance_values = model.feature_importances_
else:
    booster = model.get_booster()
    score_dict = booster.get_score(importance_type="gain")
    xgb_importance_values = np.array([
        score_dict.get(feature, 0.0) for feature in X_shap.columns
    ])

xgb_importance_df = pd.DataFrame({
    "feature": X_shap.columns,
    "xgboost_importance": xgb_importance_values,
})

importance_comparison = shap_importance_df.merge(
    xgb_importance_df,
    on="feature",
    how="inner",
)

importance_comparison["shap_importance_normalized"] = (
    importance_comparison["shap_importance"]
    / importance_comparison["shap_importance"].sum()
)

importance_comparison["xgboost_importance_normalized"] = (
    importance_comparison["xgboost_importance"]
    / importance_comparison["xgboost_importance"].sum()
)

top_features = (
    importance_comparison
    .sort_values("shap_importance", ascending=False)
    .head(15)
)

display(top_features)

In [ ]:
plot_df = top_features.melt(
    id_vars="feature",
    value_vars=[
        "shap_importance_normalized",
        "xgboost_importance_normalized",
    ],
    var_name="importance_type",
    value_name="importance_score",
)

plot_df["importance_type"] = plot_df["importance_type"].map({
    "shap_importance_normalized": "SHAP Importance",
    "xgboost_importance_normalized": "XGBoost Built-in Importance",
})

fig = px.bar(
    plot_df,
    x="feature",
    y="importance_score",
    color="importance_type",
    barmode="group",
    title="SHAP Importance vs XGBoost Built-in Importance — Top 15 Features",
    labels={
        "feature": "Feature",
        "importance_score": "Normalized Importance",
        "importance_type": "Importance Type",
    },
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
)

fig.show()

### Why SHAP Is More Reliable Than Built-in Importance

XGBoost built-in importance is useful, but it can be biased by how often a feature is used in tree splits or by the type of importance score selected.

SHAP is more reliable for business explanation because:

- It explains the actual contribution of each feature to predictions.
- It shows both direction and strength of feature impact.
- It works at both global and local levels.
- It helps explain individual fraud decisions, not only overall model structure.
- It is more suitable when model decisions need to be explained to business, risk, or compliance teams.

## 6. BUSINESS INTERPRETATION

Because `V1` to `V28` are PCA-transformed anonymized features, we cannot describe their original raw transaction meanings.

However, we can still explain them as transaction signature patterns learned from the original payment behavior.

### Plain-English interpretation of top fraud indicators

- **Feature V14 relates to the transaction signature pattern that strongly separates fraud-like transactions from normal customer behavior.**
- **Feature V17 appears to capture another high-risk transaction signature that often pushes the model toward fraud detection.**
- **Feature V12 is also a major fraud signal and may represent hidden behavioral differences between legitimate and fraudulent payments.**
- **Feature V10 contributes to fraud risk when its pattern is unusual compared with normal transactions.**
- **Feature V16 helps the model detect subtle differences in transaction structure that may not be visible through amount alone.**
- **Feature V11 may indicate patterns that reduce or increase fraud risk depending on its value and interaction with other features.**
- **Feature V4 can support fraud detection when combined with other high-impact PCA features.**
- **Feature V3 and V7 may capture additional transaction behavior differences that improve the model's risk score.**
- **Scaled Amount helps identify whether the transaction value is unusual compared with the wider dataset.**
- **Scaled Time helps capture time-based transaction behavior that may influence fraud risk.**

### Business takeaway

PayGuard does not simply flag fraud based on transaction amount.  
The model learns hidden transaction patterns from multiple anonymized features and combines them into a fraud risk score.

This makes the system more useful for real-world fraud monitoring, where fraud can appear in both small and large transaction amounts.

## 7. Explainability Summary

Key outcomes from this notebook:

- SHAP global plots identified the most important fraud indicators.
- **V14, V17, and V12** were highlighted as the strongest fraud indicators.
- Local waterfall and force plots explained one fraud case and one legitimate case.
- SHAP importance was compared with XGBoost built-in feature importance.
- SHAP was selected as the preferred explanation method because it provides clearer global and local reasoning.
- The explainability outputs make PayGuard more suitable for business, risk, and portfolio presentation use.